# Women-Led MSME Financial Access: Impact Evaluation Pipeline

**Author:** Valentina Sharma  
**Purpose:** Reproducible end-to-end pipeline simulating an RCT impact evaluation of financial access interventions on women-led micro/small businesses (MSMEs) in India.  
**Data:** Synthetic dataset generated to mirror real IE survey structure.

---

## Pipeline Overview

| Step | Script | What it does |
|------|--------|--------------|
| 1 | `01_simulate_data.py` | Generates a synthetic, intentionally messy raw survey dataset |
| 2 | `02_clean_data.py` | Deduplicates, standardises, imputes, and documents all cleaning decisions |
| 3 | `03_analysis.py` | Descriptive stats, balance checks, OLS regression, heterogeneity analysis |
| 4 | `04_visualize.py` | Produces 6 publication-quality charts from analysis outputs |

This notebook runs all steps sequentially and displays all key outputs inline.

> **Reproducibility note:** Set `np.random.seed(42)` in Step 1 ensures all results are identical on every run.

---
## Environment Setup

In [ ]:
import subprocess, sys, os

# Ensure we're running from the project root regardless of where the notebook is launched
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)
print(f"Working directory set to: {os.getcwd()}")

# Create output directories if they don't exist
for d in ["data/raw", "data/clean", "outputs/figures"]:
    os.makedirs(d, exist_ok=True)

# Core imports used throughout this notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from IPython.display import display, Markdown, Image
print("Environment ready.")

---
## Step 1 — Simulate Raw Data

We generate a synthetic dataset of **~520 women-led MSMEs** across five Indian states.

Key design choices:
- **Treatment assignment:** randomised (60% treatment / 40% control)
- **Treatment effect:** ~15% endline revenue uplift for treated firms
- **Intentional messiness:** duplicate IDs, inconsistent casing, mixed date formats, ~5% missing values — mirroring real survey data

In [ ]:
exec(open("code/01_simulate_data.py").read())

# Preview raw data
df_raw = pd.read_csv("data/raw/msme_survey_raw.csv")
print(f"\nRaw dataset: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
display(df_raw.head(8))

In [ ]:
# Show the messiness we introduced
display(Markdown("### Raw Data Quality Audit"))

audit = pd.DataFrame({
    "Missing values": df_raw.isnull().sum(),
    "Missing %":      (df_raw.isnull().mean() * 100).round(1)
})
display(audit[audit["Missing values"] > 0])

print(f"\nDuplicated business_id: {df_raw['business_id'].duplicated().sum()} rows")
print(f"Inconsistent sector values: {df_raw['sector'].unique().tolist()}")
print(f"Mixed date formats (sample): {df_raw['survey_date'].head(6).tolist()}")

---
## Step 2 — Data Cleaning

The cleaning script applies and **documents** every decision:

1. **Deduplication** — keeps last occurrence (assumed to be a correction/re-interview)
2. **Standardise categoricals** — `str.title()` on sector, education, geography
3. **Date parsing** — handles three mixed formats; ambiguous cases default to DD/MM/YYYY
4. **Missing values** — outcome variable (`endline_revenue`) flagged but NOT imputed; `employees_baseline` imputed with sector-stratified median
5. **Winsorisation** — revenue clipped at 1st–99th percentile
6. **Feature engineering** — growth rate, log revenues, state FE dummies, education numeric encoding

All decisions are written to `data/clean/cleaning_log.json` for auditability.

In [ ]:
exec(open("code/02_clean_data.py").read())

In [ ]:
# Show the cleaning log
import json
with open("data/clean/cleaning_log.json") as f:
    cleaning_log = json.load(f)

display(Markdown("### Cleaning Log (documented decisions)"))
log_df = pd.DataFrame(cleaning_log["steps"])
display(log_df[["step", "rows_affected", "detail"]])

In [ ]:
# Preview clean dataset
df_clean = pd.read_csv("data/clean/msme_survey_clean.csv")
print(f"Clean dataset: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")

display(Markdown("#### Sample of clean data (key columns)"))
key_cols = ["business_id", "state", "sector", "treatment",
            "baseline_revenue_inr", "endline_revenue_inr",
            "revenue_growth_rate", "owner_education"]
display(df_clean[key_cols].head(8))

---
## Step 3 — Analysis

### 3.1 — Descriptive Statistics

In [ ]:
exec(open("code/03_analysis.py").read())

In [ ]:
# Display descriptive stats
display(Markdown("### Descriptive Statistics — Full Sample"))
display(pd.read_csv("outputs/desc_stats_overall.csv", index_col=0).round(2))

display(Markdown("### Means by Treatment Arm"))
display(pd.read_csv("outputs/desc_stats_by_treatment.csv", index_col=0).round(2))

### 3.2 — Balance Check

A credible RCT requires that **treatment and control groups are statistically similar at baseline** — before the intervention. We run Welch's t-tests (continuous variables) and chi-square tests (binary variables). A p-value > 0.10 confirms balance.

In [ ]:
balance = pd.read_csv("outputs/balance_check.csv")
display(balance.style
    .applymap(lambda v: "color: green; font-weight: bold" if v == "YES ✓"
              else ("color: red; font-weight: bold" if v == "NO ✗" else ""),
              subset=["balanced"])
    .format({"mean_treatment": "{:.3f}", "mean_control": "{:.3f}",
             "difference": "{:+.3f}", "t_statistic": "{:.3f}", "p_value": "{:.4f}"})
    .set_caption("Balance Check Results — Welch's t-test (continuous) / Chi-square (binary)")
)

### 3.3 — Main OLS Regression

**Specification:**
$$\log(\text{EndlineRevenue}_i) = \alpha + \beta \cdot \text{Treatment}_i + \gamma \cdot \log(\text{BaselineRevenue}_i) + \delta \cdot \mathbf{X}_i + \lambda_s + \varepsilon_i$$

Where:
- $\beta$ = **ITT (Intent-to-Treat) estimate** — the causal effect of treatment assignment
- $\mathbf{X}_i$ = baseline controls (employees, loan access, business age)
- $\lambda_s$ = state fixed effects
- Standard errors: **HC3 heteroskedasticity-robust**

In [ ]:
reg = pd.read_csv("outputs/regression_results.csv", index_col=0)

# Show only core variables (hide state FEs for readability)
core_vars = reg[~reg.index.str.startswith("state_fe_")]
display(core_vars[["coefficient", "std_error", "t_statistic", "p_value",
                   "ci_lower", "ci_upper", "significant"]]
    .style
    .applymap(lambda v: "font-weight: bold; color: #2166AC"
              if v in ["***", "**", "*"] else "", subset=["significant"])
    .format({"coefficient": "{:.4f}", "std_error": "{:.4f}",
             "t_statistic": "{:.3f}", "p_value": "{:.4f}",
             "ci_lower": "{:.4f}", "ci_upper": "{:.4f}"})
    .set_caption("OLS Regression Results — Dependent variable: log(endline revenue). HC3 robust SEs.")
)

# Key finding
t_coef = reg.loc["treatment", "coefficient"]
t_pval = reg.loc["treatment", "p_value"]
effect_pct = (np.exp(t_coef) - 1) * 100
print(f"\n── KEY FINDING ───────────────────────────────────────")
print(f"Treatment coefficient β = {t_coef:.4f}  (p = {t_pval:.4f})")
print(f"Implied effect: treated firms have ~{effect_pct:.1f}% higher endline revenue")
print(f"──────────────────────────────────────────────────────")

### 3.4 — Heterogeneity Analysis

We test whether the treatment effect varies across two key dimensions:
- **Baseline loan access** — do previously banked firms respond differently?
- **Owner education** — do graduate-owned firms benefit more?

In [ ]:
hetero = pd.read_csv("outputs/heterogeneity_results.csv")

# Compute implied % effects
hetero["base_effect_pct"]      = (np.exp(hetero["base_treatment_effect"]) - 1) * 100
hetero["subgroup_effect_pct"]  = (np.exp(hetero["base_treatment_effect"] + hetero["interaction_coef"]) - 1) * 100
hetero["interaction_pct"]      = (np.exp(hetero["interaction_coef"]) - 1) * 100

display(hetero[["subgroup", "base_effect_pct", "subgroup_effect_pct",
                "interaction_coef", "p_interaction"]]
    .rename(columns={
        "subgroup": "Subgroup",
        "base_effect_pct": "Effect (base group) %",
        "subgroup_effect_pct": "Effect (subgroup) %",
        "interaction_coef": "Interaction coef",
        "p_interaction": "p (interaction)"
    })
    .round(3)
    .set_index("Subgroup")
    .style.set_caption("Heterogeneity Analysis — Interaction term OLS")
)

---
## Step 4 — Visualisations

In [ ]:
exec(open("code/04_visualize.py").read())

In [ ]:
# Display all figures inline
from IPython.display import Image, display

figures = [
    ("outputs/figures/fig1_revenue_distribution.png",   "Fig 1 — Revenue Distribution: Treatment vs Control"),
    ("outputs/figures/fig2_balance_check.png",          "Fig 2 — Balance Check"),
    ("outputs/figures/fig3_regression_coefficients.png","Fig 3 — OLS Regression Coefficients"),
    ("outputs/figures/fig4_heterogeneity.png",          "Fig 4 — Heterogeneity Analysis"),
    ("outputs/figures/fig5_sector_composition.png",     "Fig 5 — Sector Composition by Treatment Arm"),
    ("outputs/figures/fig6_revenue_growth_violin.png",  "Fig 6 — Revenue Growth Rate Distribution"),
]

for path, caption in figures:
    display(Markdown(f"### {caption}"))
    display(Image(filename=path, width=820))

---
## Summary of Findings

This simulated impact evaluation demonstrates a full reproducible pipeline for evaluating financial access interventions on women-led MSMEs. Key results:

In [ ]:
# Auto-generate a findings summary from outputs
reg     = pd.read_csv("outputs/regression_results.csv", index_col=0)
balance = pd.read_csv("outputs/balance_check.csv")
df_clean = pd.read_csv("data/clean/msme_survey_clean.csv")
df_reg   = df_clean[df_clean["endline_revenue_missing"] == 0]

t_coef      = reg.loc["treatment", "coefficient"]
t_pval      = reg.loc["treatment", "p_value"]
effect_pct  = (np.exp(t_coef) - 1) * 100
n_balanced  = (balance["balanced"] == "YES ✓").sum()
n_total     = len(balance)
n_obs       = len(df_reg)
n_treat     = df_reg["treatment"].sum()
n_ctrl      = (df_reg["treatment"] == 0).sum()
med_growth_t = df_reg[df_reg["treatment"]==1]["revenue_growth_rate"].median()
med_growth_c = df_reg[df_reg["treatment"]==0]["revenue_growth_rate"].median()

summary_md = f"""
| Finding | Value |
|---------|-------|
| Sample size (regression) | {n_obs} firms ({n_treat} treatment, {n_ctrl} control) |
| Balance check | {n_balanced}/{n_total} variables balanced at baseline |
| Treatment effect (ITT β) | {t_coef:+.4f} log points (p = {t_pval:.4f}) |
| Implied revenue effect | **+{effect_pct:.1f}%** endline revenue for treated firms |
| Median growth rate — treatment | {med_growth_t:.1%} |
| Median growth rate — control   | {med_growth_c:.1%} |
"""
display(Markdown(summary_md))

sig_str = "statistically significant" if t_pval < 0.05 else "not statistically significant at 5%"
print(f"\nInterpretation: Treatment assignment is associated with a {effect_pct:.1f}% increase in")
print(f"endline revenue ({sig_str}, p={t_pval:.4f}), controlling for baseline revenue,")
print(f"employees, loan access, business age, and state fixed effects.")

---
## Outputs Produced

All outputs are saved and ready for replication:

**Data**
- `data/raw/msme_survey_raw.csv` — original (messy) simulated survey data
- `data/clean/msme_survey_clean.csv` — analysis-ready clean dataset
- `data/clean/cleaning_log.json` — full audit trail of all cleaning decisions

**Analysis outputs**
- `outputs/desc_stats_overall.csv`
- `outputs/desc_stats_by_treatment.csv`
- `outputs/sector_distribution.csv`
- `outputs/balance_check.csv`
- `outputs/regression_results.csv`
- `outputs/heterogeneity_results.csv`

**Figures**
- `outputs/figures/fig1_revenue_distribution.png`
- `outputs/figures/fig2_balance_check.png`
- `outputs/figures/fig3_regression_coefficients.png`
- `outputs/figures/fig4_heterogeneity.png`
- `outputs/figures/fig5_sector_composition.png`
- `outputs/figures/fig6_revenue_growth_violin.png`
- `outputs/figures/fig7_summary_panel.png`

---
*This notebook is part of the Women-Led MSME Impact Evaluation pipeline. See `README.md` for full documentation.*